# NLR Fraud Detection — EDA (Synthetic Data)

This notebook generates **synthetic** card-not-present style transactions, compares fraud vs. legitimate distributions, checks class imbalance, and writes a CSV snapshot for training pipelines.

**Setup:** from the repository root, install the package in editable mode:

`pip install -e ".[dev]"` or `pip install -r requirements-dev.txt && pip install -e .`

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from nlr_fraud.config import load_config
from nlr_fraud.features import add_log_amount, add_night_flag
from nlr_fraud.synthetic_data import generate_transactions

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4)

ROOT = Path("..").resolve()
CFG = load_config(ROOT / "configs" / "default.yaml")
CFG

In [ ]:
df = generate_transactions(
    n_samples=CFG.data.n_samples,
    fraud_rate=CFG.data.fraud_rate,
    random_seed=CFG.data.random_seed,
)
df.head()

## Schema and basic quality checks

Confirm row counts, label balance, missing values, and simple ranges for bounded fields.

In [ ]:
summary = pd.DataFrame(
    {
        "rows": [len(df)],
        "fraud_rate": [df["is_fraud"].mean()],
        "dup_ids": [df["transaction_id"].duplicated().sum()],
        "missing_cells": [int(df.isna().sum().sum())],
    }
)
summary

In [ ]:
df.describe(include="all").T

## Univariate comparisons (fraud vs. legitimate)

Fraudulent rows are simulated with higher amounts, riskier merchants, more new devices, more country mismatches, and higher short-window velocity.

In [ ]:
num_cols = [
    "amount",
    "merchant_risk_score",
    "txn_count_24h",
    "hour_of_day",
]
plot_df = df.copy()
plot_df["label"] = plot_df["is_fraud"].map({0: "legitimate", 1: "fraud"})

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.ravel()
for ax, col in zip(axes, num_cols):
    sns.kdeplot(data=plot_df, x=col, hue="label", common_norm=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
cat_cols = ["device_new_to_user", "country_mismatch"]
counts = (
    df.groupby("is_fraud")[cat_cols]
    .mean()
    .rename(index={0: "legitimate", 1: "fraud"})
)
counts

## Engineered previews (not required for baseline training)

Log amount and a simple night flag illustrate how domain transforms can be layered before model fitting.

In [ ]:
enriched = add_night_flag(add_log_amount(df))
enriched[["amount", "log_amount", "is_night", "is_fraud"]].head()

## Correlation structure

Inspect linear associations among numeric risk signals (interpret cautiously with synthetic data).

In [ ]:
corr = df.drop(columns=["transaction_id"]).corr(numeric_only=True)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Correlation matrix")
plt.tight_layout()

## Persist snapshot for training

Writes `data/raw/synthetic_transactions.csv` (gitignored) for reproducible offline training.

In [ ]:
out_path = ROOT / "data" / "raw" / "synthetic_transactions.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
out_path, out_path.stat().st_size